# Projeto de Grupo — Tópicos Avançados em Data Mining
**Mestrado em Ciência de Dados e Gestão — 2026/2027**

**Análise de comentários Airbnb e classificação de anúncios de alta qualidade — Cidade do Cabo (Cape Town)**

Dados: [Inside Airbnb](https://insideairbnb.com/get-the-data/) — ficheiros detalhados `reviews.csv.gz` e `listings.csv.gz` (scrape de julho de 2026).

## Índice
- **0. Setup**
- **Parte I — Text Mining sobre os comentários dos hóspedes**
  1. Compreensão e documentação do dataset
  2. Limpeza e preparação de dados
  3. Análise de sentimentos
  4. Modelação de tópicos
  5. Comparação com a literatura
  6. Construção das variáveis de texto
- **Parte II — Modelos supervisionados de classificação de anúncios**
  1. Integração e preparação dos dados
  2. Naive Bayes
  3. Random Forest
  4. Support Vector Machine
  5. Contributo das variáveis de texto
  6. Conclusão e comparação de modelos
- **Conclusões gerais e limitações**

## 0. Setup
Bibliotecas, caminhos e parâmetros globais. As dependências estão em `requirements.txt`.

Estrutura de pastas:
- `data/` — ficheiros originais do Inside Airbnb
- `outputs/figures/` — gráficos exportados (reutilizados na apresentação)
- `outputs/intermediate/` — ficheiros intermédios da Parte I (entregáveis)

In [ ]:
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

warnings.filterwarnings("ignore")

# Caminhos
DATA_DIR = Path("data")
FIG_DIR = Path("outputs/figures")
INT_DIR = Path("outputs/intermediate")
FIG_DIR.mkdir(parents=True, exist_ok=True)
INT_DIR.mkdir(parents=True, exist_ok=True)

# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo dos gráficos
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4.5)
pd.set_option("display.max_colwidth", 120)

def save_fig(name):
    """Guarda a figura atual em outputs/figures para uso na apresentação."""
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

---
# PARTE I — Text Mining sobre os comentários dos hóspedes

## 1. Compreensão e Documentação do Dataset

### 1.1 Escolha da cidade e descrição do conjunto de reviews

**Porquê Cape Town?**
- **Dimensão do corpus:** com cerca de 760 mil comentários em mais de 22 mil anúncios, é um dos mercados com mais reviews no Inside Airbnb. Isso dá dados suficientes para a modelação de tópicos e para treinar modelos supervisionados.
- **Interesse do mercado:** é um dos principais destinos turísticos de África e do hemisfério sul, com uma oferta muito variada (apartamentos no centro, casas de praia, alojamento de luxo, quartos privados).
- **Idioma:** a maioria dos hóspedes escreve em inglês, a língua com melhor suporte nas ferramentas de PLN usadas (VADER, modelos BERT, BERTopic). Mesmo assim, o mercado internacional traz diversidade linguística que será caracterizada na secção 2.

In [ ]:
reviews = pd.read_csv(DATA_DIR / "reviews.csv.gz", parse_dates=["date"])
listings = pd.read_csv(DATA_DIR / "listings.csv.gz", low_memory=False)

print(f"Reviews:  {reviews.shape[0]:,} linhas x {reviews.shape[1]} colunas")
print(f"Listings: {listings.shape[0]:,} linhas x {listings.shape[1]} colunas")
reviews.head()

In [ ]:
# Descrição dos campos do ficheiro de reviews
field_desc = {
    "listing_id": "Identificador do anúncio (chave de ligação ao listings.csv)",
    "id": "Identificador único da review",
    "date": "Data da review",
    "reviewer_id": "Identificador do hóspede",
    "reviewer_name": "Primeiro nome do hóspede",
    "comments": "Texto livre do comentário (objeto da análise)",
}
pd.DataFrame({
    "tipo": reviews.dtypes.astype(str),
    "n_únicos": reviews.nunique(),
    "descrição": pd.Series(field_desc),
})

In [ ]:
# Período temporal coberto e cobertura de anúncios
print(f"Primeira review: {reviews['date'].min().date()}")
print(f"Última review:   {reviews['date'].max().date()}")
print(f"Anúncios com pelo menos uma review: {reviews['listing_id'].nunique():,} "
      f"de {listings['id'].nunique():,} anúncios no listings.csv "
      f"({reviews['listing_id'].nunique() / listings['id'].nunique():.1%})")
print(f"Hóspedes distintos: {reviews['reviewer_id'].nunique():,}")

per_year = reviews["date"].dt.year.value_counts().sort_index()
ax = per_year.plot(kind="bar", color="C0")
ax.set(title="Número de reviews por ano", xlabel="Ano", ylabel="Nº de reviews")
save_fig("p1_reviews_per_year")
plt.show()

### 1.2 Qualidade dos dados
Analisamos:
1. **Valores vazios** em cada campo;
2. **Duplicados**: IDs repetidos, linhas idênticas e o mesmo comentário repetido no mesmo anúncio;
3. **Comentários automáticos da plataforma**, por exemplo *"The host canceled this reservation X days before arrival. This is an automated posting."*;
4. **Comentários sem conteúdo textual relevante**: sem letras (só pontuação ou símbolos, ex.: `"."`, `"-"`) ou muito curtos;
5. **Ruído de formatação**: tags HTML (`<br/>`) dentro do texto.

Nesta fase só **identificamos e quantificamos** estes problemas. O tratamento é decidido na secção 2 (Limpeza e Preparação).

In [ ]:
# 1) Valores vazios
missing = pd.DataFrame({
    "n_vazios": reviews.isna().sum(),
    "% vazios": (reviews.isna().mean() * 100).round(3),
})
missing

In [ ]:
# Texto auxiliar: comentários sem NaN e sem tags HTML, para as contagens seguintes
text = reviews["comments"].fillna("")
text_plain = text.str.replace(r"<br\s*/?>", " ", regex=True).str.strip()

# 2) Duplicados
dup = {
    "IDs de review duplicados": reviews["id"].duplicated().sum(),
    "Linhas totalmente idênticas": reviews.duplicated().sum(),
    "Mesmo hóspede + anúncio + comentário": reviews.duplicated(["reviewer_id", "listing_id", "comments"]).sum(),
    "Mesmo anúncio + comentário (qualquer hóspede)": reviews.dropna(subset=["comments"]).duplicated(["listing_id", "comments"]).sum(),
}
pd.Series(dup, name="n").to_frame()

In [ ]:
# Exemplos de comentários duplicados no mesmo anúncio: na maioria são textos genéricos curtos
dups = reviews.dropna(subset=["comments"])
dups = dups[dups.duplicated(["listing_id", "comments"], keep=False)]
dups["comments"].value_counts().head(10)

In [ ]:
# 3) Comentários automáticos da plataforma (cancelamentos)
AUTO_PATTERN = r"automated posting|the host canceled this reservation"
is_auto = text_plain.str.contains(AUTO_PATTERN, case=False, regex=True)
print(f"Comentários automáticos detetados: {is_auto.sum():,}")
text_plain[is_auto].head()

In [ ]:
# 4) Comentários sem conteúdo textual relevante
n_words = text_plain.str.split().str.len().fillna(0).astype(int)
# Usamos o módulo re do Python (Unicode): o regex do backend pyarrow do pandas só reconhece letras ASCII em \W
LETTER_RE = re.compile(r"[^\W\d_]")
has_letters = text_plain.map(lambda s: bool(LETTER_RE.search(s)))   # pelo menos uma letra (qualquer alfabeto)

quality = pd.DataFrame({
    "vazio (NaN ou só espaços)": text_plain.eq(""),
    "sem letras (só pontuação/símbolos)": text_plain.ne("") & ~has_letters,
    "muito curto (1-2 palavras)": has_letters & n_words.between(1, 2),
    "com tags HTML <br/>": text.str.contains(r"<br\s*/?>", regex=True),
})
summary = pd.DataFrame({"n": quality.sum(), "%": (quality.mean() * 100).round(2)})
summary

In [ ]:
print("Exemplos de comentários sem letras:")
print(text_plain[quality["sem letras (só pontuação/símbolos)"]].value_counts().head(10).to_string())
print("\nComentários muito curtos (1-2 palavras) mais frequentes:")
print(text_plain[quality["muito curto (1-2 palavras)"]].value_counts().head(15).to_string())

### 1.3 Estatísticas descritivas

In [ ]:
# Comprimento dos comentários (só comentários com texto)
valid = text_plain.ne("")
length = pd.DataFrame({
    "caracteres": text_plain[valid].str.len(),
    "palavras": n_words[valid],
})
length.describe().T.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(length["palavras"].clip(upper=300), bins=60, ax=axes[0])
axes[0].axvline(length["palavras"].mean(), color="red", ls="--", label=f"média = {length['palavras'].mean():.0f}")
axes[0].axvline(length["palavras"].median(), color="black", ls=":", label=f"mediana = {length['palavras'].median():.0f}")
axes[0].set(title="Comprimento dos comentários (palavras, truncado a 300)", xlabel="Nº de palavras", ylabel="Nº de reviews")
axes[0].legend()

sns.histplot(np.log10(length["palavras"].clip(lower=1)), bins=60, ax=axes[1])
axes[1].set(title="Comprimento dos comentários (escala log10)", xlabel="log10(nº de palavras)", ylabel="Nº de reviews")
save_fig("p1_comment_length")
plt.show()

In [ ]:
# Distribuição do número de reviews por anúncio
per_listing = reviews.groupby("listing_id").size().rename("n_reviews")
display(per_listing.describe().round(1).to_frame().T)

bins = [0, 1, 4, 9, 19, 49, 99, np.inf]
labels = ["1", "2-4", "5-9", "10-19", "20-49", "50-99", "100+"]
buckets = pd.cut(per_listing, bins=bins, labels=labels).value_counts().sort_index()
display(pd.DataFrame({"n_anúncios": buckets, "%": (buckets / buckets.sum() * 100).round(1)}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
buckets.plot(kind="bar", ax=axes[0], color="C0")
axes[0].set(title="Anúncios por nº de reviews", xlabel="Nº de reviews", ylabel="Nº de anúncios")
axes[0].tick_params(axis="x", rotation=0)

# Curva de concentração: que % das reviews vem dos X% anúncios com mais reviews
cum = per_listing.sort_values(ascending=False).cumsum() / per_listing.sum()
axes[1].plot(np.arange(1, len(cum) + 1) / len(cum) * 100, cum.values * 100)
axes[1].set(title="Concentração das reviews", xlabel="% de anúncios (ordenados por nº de reviews)", ylabel="% acumulada de reviews")
save_fig("p1_reviews_per_listing")
plt.show()

top20 = per_listing.sort_values(ascending=False).head(int(len(per_listing) * 0.2)).sum() / per_listing.sum()
print(f"Os 20% de anúncios com mais reviews concentram {top20:.1%} de todas as reviews.")

**Palavras mais frequentes.** Contagem simples: tokenização por letras, minúsculas, sem *stopwords* inglesas. É uma visão preliminar, feita antes da filtragem por idioma (secção 2), por isso podem aparecer palavras de outros idiomas.

In [ ]:
STOP = set(ENGLISH_STOP_WORDS)
token_re = re.compile(r"[a-z]+(?:'[a-z]+)?")

word_counts = Counter()
for doc in text_plain[valid].str.lower():
    word_counts.update(w for w in token_re.findall(doc) if w not in STOP and len(w) > 2)

top_words = pd.DataFrame(word_counts.most_common(30), columns=["palavra", "frequência"])
ax = top_words.set_index("palavra")["frequência"][::-1].plot(kind="barh", figsize=(8, 8), color="C0")
ax.set(title="30 palavras mais frequentes (sem stopwords)", xlabel="Frequência", ylabel="")
save_fig("p1_top_words")
plt.show()

In [ ]:
from wordcloud import WordCloud

wc = WordCloud(width=1200, height=500, background_color="white", random_state=RANDOM_STATE)
wc.generate_from_frequencies(dict(word_counts.most_common(200)))
plt.figure(figsize=(12, 5))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Nuvem de palavras dos comentários")
save_fig("p1_wordcloud")
plt.show()

### 1.4 Síntese da secção 1
**Dimensão e período.** O corpus tem **760 023 reviews** de **511 074 hóspedes** distintos, sobre **22 322 anúncios**, entre **junho de 2010 e julho de 2026**. O volume cresce quase todos os anos: há uma quebra clara em 2020 (COVID-19), e desde 2022 o crescimento é forte (2025 é o ano com mais reviews). Dos 27 381 anúncios do `listings.csv`, **18,5% não têm nenhuma review**, uma questão a tratar na Parte II.

**Qualidade dos dados.**
| Problema | N | % | Tratamento previsto (secção 2) |
|---|---|---|---|
| Comentário vazio (NaN) | 158 | 0,02% | Remover |
| Linhas ou IDs duplicados | 0 | 0% | — |
| Mesmo hóspede + anúncio + comentário | 244 | 0,03% | Remover (duplicado real) |
| Mesmo comentário no mesmo anúncio (hóspedes diferentes) | 771 | 0,10% | Manter: são sobretudo textos genéricos ("Great place", ".") escritos por hóspedes diferentes |
| Comentários automáticos da plataforma | 0 | 0% | — (esta versão do Inside Airbnb já não inclui as mensagens automáticas de cancelamento) |
| Sem letras (".", "-", emojis, "10/10") | 2 068 | 0,27% | Remover: sem conteúdo para PLN |
| Muito curtos (1-2 palavras) | 22 358 | 2,94% | Manter para sentimento; excluir da modelação de tópicos |
| Tags HTML `<br/>` | 159 661 | 21% | Substituir por espaço |

No geral a qualidade é **boa**: os problemas reais afetam menos de 0,5% das reviews.

**Estatísticas descritivas.**
- **Comprimento:** em média **39 palavras** (224 caracteres), com mediana de 28. A distribuição é muito assimétrica à direita, com comentários de até 1 000 palavras. A maioria é curta, o que favorece métodos pensados para textos curtos (VADER, BERTopic) face ao LDA clássico.
- **Reviews por anúncio:** a média é 34 e a mediana 13, uma distribuição de cauda longa. **27,4% dos anúncios têm menos de 5 reviews** e os **20% de anúncios com mais reviews concentram 69% das reviews**. Na Parte II isto justifica um número mínimo de reviews para a variável *target* e o cuidado com variáveis agregadas de anúncios com poucos comentários.
- **Palavras frequentes:** dominam palavras positivas (*great, beautiful, clean, lovely, amazing, perfect*) e palavras dos temas clássicos da satisfação do turista: **localização** (*location, view(s), town*), **anfitrião** (*host, friendly, helpful*), **limpeza e conforto** (*clean, comfortable*) e **tipo de alojamento** (*apartment, house, home*). Aparece também *und* (alemão), sinal de que o corpus é multilingue e de que é preciso filtrar por idioma (secção 2).

## 2. Limpeza e Preparação de Dados

O pipeline desta secção tem quatro passos:
1. **Limpeza base** e remoção dos registos inválidos identificados na secção 1;
2. **Deteção de idioma** e escolha do idioma de análise;
3. **Normalização do texto** em dois níveis, consoante o método que o vai usar;
4. **Estratégia de amostragem**, tendo em conta o custo computacional de cada método.

### 2.1 Limpeza base e remoção de registos inválidos
A limpeza base é **conservadora**: só remove ruído de formatação, sem alterar o conteúdo linguístico.
- `<br/>` e quebras de linha → espaço; entidades HTML (`&amp;`, `&#39;`) → caracteres normais;
- URLs removidos (não têm valor semântico e confundem a deteção de idioma);
- espaços múltiplos colapsados.

Mantemos **maiúsculas, pontuação e emojis**, porque o VADER e os modelos transformer os usam como sinais de intensidade ("GREAT!!!", "😍"). O resultado é a coluna `text_clean`.

Registos removidos, com base no diagnóstico da secção 1:
- comentários vazios e comentários sem letras (".", "-", só emojis), que não têm conteúdo para PLN;
- duplicados reais (mesmo hóspede, mesmo anúncio, mesmo texto).

In [ ]:
import html

def clean_text(s: pd.Series) -> pd.Series:
    """Limpeza base: remove ruído de formatação, preserva maiúsculas, pontuação e emojis."""
    return (s.fillna("")
             .str.replace(r"<br\s*/?>", " ", regex=True)
             .map(html.unescape)
             .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())

df = reviews.copy()
df["text_clean"] = clean_text(df["comments"])

funnel = [("Reviews originais", len(df))]
df = df[df["text_clean"].ne("")]
funnel.append(("- comentários vazios", len(df)))
df = df[df["text_clean"].map(lambda s: bool(LETTER_RE.search(s)))]
funnel.append(("- comentários sem letras", len(df)))
df = df.drop_duplicates(["reviewer_id", "listing_id", "comments"])
funnel.append(("- duplicados (hóspede + anúncio + texto)", len(df)))

def show_funnel(funnel):
    f = pd.DataFrame(funnel, columns=["passo", "n_reviews"])
    f["removidas"] = (-f["n_reviews"].diff()).fillna(0).astype(int)
    f["% do original"] = (f["n_reviews"] / f["n_reviews"].iloc[0] * 100).round(2)
    return f

show_funnel(funnel)

### 2.2 Identificação do idioma
Usamos o **[lingua](https://github.com/pemistahl/lingua-py)** em vez do `langdetect`, que é mais comum. Num teste preliminar com 5 000 comentários:
- o lingua foi **~3× mais rápido** (implementação em Rust, com paralelização);
- foi **mais fiável em textos curtos**, que são muitos neste corpus: o `langdetect` classificava "Nice place" como romeno e "Feels like home." como afrikaans, enquanto o lingua acertou no inglês.

A deteção sobre ~758 mil comentários demora alguns minutos, por isso o resultado fica guardado em `outputs/intermediate/reviews_language.parquet` e é reutilizado nas execuções seguintes.

In [ ]:
from lingua import LanguageDetectorBuilder

LANG_CACHE = INT_DIR / "reviews_language.parquet"

if LANG_CACHE.exists():
    lang = pd.read_parquet(LANG_CACHE)
else:
    detector = (LanguageDetectorBuilder.from_all_languages()
                .with_preloaded_language_models().build())
    detected = detector.detect_languages_in_parallel_of(df["text_clean"].tolist())
    lang = pd.DataFrame({
        "id": df["id"].values,
        "language": [d.iso_code_639_1.name.lower() if d else "unknown" for d in detected],
        "language_name": [d.name.title() if d else "Unknown" for d in detected],
    })
    lang.to_parquet(LANG_CACHE, index=False)

df = df.merge(lang, on="id", how="left")
df["n_words"] = df["text_clean"].str.split().str.len()

# Correção para textos muito curtos: o lingua classifica mal palavras isoladas
# ("Good" -> somali, "Nice" -> checo, "Perfect" -> latim). Se um comentário com <= 3 palavras
# contém apenas palavras do vocabulário inglês (WordNet + stopwords), é reclassificado como inglês.
import nltk
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords, wordnet

EN_VOCAB = {w.lower() for w in wordnet.all_lemma_names()} | set(stopwords.words("english"))
alpha_re = re.compile(r"[a-z]+")

def all_english_words(s: str) -> bool:
    tokens = alpha_re.findall(s.lower())
    return len(tokens) > 0 and all(t in EN_VOCAB for t in tokens)

df["language_raw"] = df["language"]
fix = (df["n_words"] <= 3) & (df["language"] != "en") & df["text_clean"].map(all_english_words)
df.loc[fix, ["language", "language_name"]] = ["en", "English"]
print(f"Comentários curtos reclassificados como inglês: {fix.sum():,}")
print(df.loc[fix, ["text_clean", "language_raw"]].value_counts().head(8).to_string(), "\n")

lang_dist = (df["language_name"].value_counts().rename("n_reviews").to_frame()
             .assign(**{"%": lambda d: (d["n_reviews"] / d["n_reviews"].sum() * 100).round(2)}))
print(f"Idiomas distintos detetados: {len(lang_dist)}")
lang_dist.head(15)

In [ ]:
top = lang_dist["n_reviews"].head(10)
others = pd.Series({"Outros": lang_dist["n_reviews"].iloc[10:].sum()})
plot_data = pd.concat([top, others])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
(plot_data / plot_data.sum() * 100)[::-1].plot(kind="barh", ax=axes[0], color="C0")
axes[0].set(title="Distribuição dos idiomas (% das reviews)", xlabel="% das reviews", ylabel="")

# Peso do inglês por comprimento do comentário: a deteção é menos fiável em textos muito curtos
len_bins = pd.cut(df["n_words"], [0, 2, 5, 10, 25, 50, np.inf], labels=["1-2", "3-5", "6-10", "11-25", "26-50", "50+"])
en_by_len = df.groupby(len_bins, observed=True)["language"].apply(lambda s: (s == "en").mean() * 100)
en_by_len.plot(kind="bar", ax=axes[1], color="C1")
axes[1].set(title="% de reviews em inglês por comprimento", xlabel="Nº de palavras", ylabel="% inglês", ylim=(0, 100))
axes[1].tick_params(axis="x", rotation=0)
save_fig("p1_language_distribution")
plt.show()

In [ ]:
# Exemplos por idioma, para validar a deteção qualitativamente
examples = (df[df["language"].isin(["de", "fr", "nl", "pt", "af", "es"])]
            .query("n_words >= 4")
            .groupby("language_name").sample(2, random_state=RANDOM_STATE)[["language_name", "text_clean"]])
examples.assign(text_clean=examples["text_clean"].str[:110])

In [ ]:
# Casos potencialmente mal classificados: comentários muito curtos detetados como não-inglês
# (após a correção) comentários curtos que continuam como não-inglês
short_non_en = df[(df["n_words"] <= 2) & (df["language"] != "en")]
print(f"Comentários com 1-2 palavras ainda classificados como não-inglês: {len(short_non_en):,}")
short_non_en[["text_clean", "language_name"]].value_counts().head(12)

### 2.3 Escolha do idioma de análise
Analisamos **apenas os comentários em inglês**:
- é o idioma **largamente dominante** no corpus (ver tabela acima), por isso a análise mantém a grande maioria da informação;
- as ferramentas de sentimento (VADER é *só* para inglês; o RoBERTa da Cardiff NLP foi treinado em inglês) e de tópicos (embeddings `all-MiniLM-L6-v2`, *stopwords*, lematização) têm o melhor suporte e desempenho em inglês;
- misturar idiomas na modelação de tópicos geraria **tópicos por idioma** em vez de tópicos por tema.

**Custo desta decisão:** perdem-se os comentários noutros idiomas, sobretudo de hóspedes alemães, franceses e neerlandeses. Isto pode enviesar ligeiramente as variáveis por anúncio em anúncios muito procurados por esses mercados. Os anúncios cujos comentários são todos não-ingleses ficam sem variáveis de texto, uma situação tratada na Parte II §1.

Nota: nos comentários muito curtos a deteção automática é pouco fiável. Corrigimo-la com uma regra simples (só palavras do vocabulário inglês → inglês). Os casos ambíguos que sobram ("Super", "Top", "Alles gut") são poucos e têm pouca informação, por isso o impacto é desprezável.

In [ ]:
en = df[df["language"] == "en"].copy()
funnel.append(("- comentários não ingleses", len(en)))
show_funnel(funnel)

In [ ]:
print(f"Reviews em inglês: {len(en):,}")
print(f"Anúncios com pelo menos 1 review em inglês: {en['listing_id'].nunique():,} "
      f"de {reviews['listing_id'].nunique():,} com reviews")
lost = set(reviews["listing_id"]) - set(en["listing_id"])
print(f"Anúncios que perdem todas as reviews com o filtro de idioma: {len(lost):,}")

### 2.4 Normalização do texto (dois níveis)
Cada método precisa de um nível de normalização diferente, por isso mantemos **duas versões do texto**:

| Coluna | Usada em | Transformações | Justificação |
|---|---|---|---|
| `text_clean` | VADER, RoBERTa, embeddings do BERTopic | só limpeza base (2.1) | Estes modelos usam maiúsculas, pontuação, emojis e negações ("not clean"); removê-los degrada os resultados |
| `text_topic` | LDA, vocabulário do BERTopic, contagens de palavras | minúsculas; só tokens alfabéticos; remoção de *stopwords*; **remoção do nome do anfitrião**; lematização (WordNet); tokens com mais de 2 letras | O LDA é um modelo *bag-of-words*: sem esta normalização os tópicos ficam dominados por palavras funcionais e variantes morfológicas |

Decisões específicas:
- **Nomes dos anfitriões:** aparecem em muitas reviews ("Thanks Katrin!") e, num modelo *bag-of-words*, criariam tópicos por anfitrião. Removemo-los usando o campo `host_name` do `listings.csv`.
- ***Stopwords* de domínio:** além das *stopwords* gerais, removemos termos presentes em quase todos os comentários e sem valor discriminativo entre temas (*stay, place, airbnb, cape, town*). A lista pode ser revista na modelação de tópicos (secção 4).
- **Lematização em vez de *stemming*:** mantém palavras legíveis (*views → view*, enquanto o *stemming* daria *beauti*), o que facilita interpretar os tópicos.
- **Negações:** o `text_topic` perde a negação ("not clean" → "clean"). Por isso **o sentimento é sempre calculado sobre o `text_clean`**.

In [ ]:
import nltk
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
from functools import lru_cache
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

DOMAIN_STOP = {"stay", "stayed", "staying", "place", "airbnb", "cape", "town", "would", "also", "us"}
STOP_TOPIC = set(stopwords.words("english")) | set(ENGLISH_STOP_WORDS) | DOMAIN_STOP

lemmatizer = WordNetLemmatizer()
lemma = lru_cache(maxsize=None)(lemmatizer.lemmatize)
word_re = re.compile(r"[a-z]+")

# Tokens do nome do anfitrião por anúncio (ex.: "Mark And Tracey" -> {"mark", "tracey"})
host_tokens = (listings.set_index("id")["host_name"].fillna("").str.lower()
               .map(lambda s: frozenset(word_re.findall(s)) - STOP_TOPIC))

def normalize_for_topics(text: str, listing_id: int) -> str:
    """Normalização para modelos bag-of-words (LDA): ver tabela acima."""
    hosts = host_tokens.get(listing_id, frozenset())
    tokens = (lemma(w) for w in word_re.findall(text.lower()) if w not in STOP_TOPIC and w not in hosts)
    return " ".join(t for t in tokens if len(t) > 2 and t not in STOP_TOPIC)

# Exemplo
ex = en[en["n_words"].between(25, 40)].sample(3, random_state=RANDOM_STATE)
for _, row in ex.iterrows():
    print("ORIGINAL :", row["text_clean"])
    print("TOPIC    :", normalize_for_topics(row["text_clean"], row["listing_id"]), "\n")

### 2.5 Estratégia de amostragem
Os métodos têm custos computacionais muito diferentes. Medimos o *throughput* neste computador (Apple M2, GPU via MPS):

| Método | Velocidade aproximada | Tempo para todo o corpus inglês |
|---|---|---|
| VADER (léxico) | milhares de comentários/s | segundos a poucos minutos |
| RoBERTa (`cardiffnlp/twitter-roberta-base-sentiment-latest`) | ~45 comentários/s (~110/s ordenando por comprimento) | **~4 horas** |
| Embeddings MiniLM (BERTopic) | ~300 comentários/s | ~40 minutos |
| LDA com pesquisa do nº de tópicos (vários *k* × coerência) | muitas execuções do modelo | impraticável no corpus inteiro |

Por isso usamos uma **amostragem em três níveis**:

1. **Nível 1 — Corpus inglês completo → VADER.** É barato, por isso não há razão para amostrar.
2. **Nível 2 — Corpus de variáveis → RoBERTa + atribuição de tópicos.** São **as 20 reviews inglesas mais recentes de cada anúncio**. Porquê:
   - **cobre 100% dos anúncios** com reviews em inglês, o que é essencial porque a secção 6 agrega por anúncio;
   - a mediana é de 13 reviews por anúncio, por isso **a maioria dos anúncios fica com todas as reviews**; só os anúncios muito populares são truncados;
   - impede que os ~20% de anúncios que concentram 69% das reviews dominem as análises ao nível da review;
   - as reviews mais recentes descrevem melhor o estado **atual** do anúncio, o mesmo período que mais pesa no `review_scores_rating`;
   - reduz o custo do RoBERTa de ~4 h para menos de 1 h.
3. **Nível 3 — Amostra de modelação → ajuste do LDA e do BERTopic, escolha de *k* e coerência.** São **30 000 reviews** aleatórias do Nível 2, com pelo menos 5 tokens no `text_topic` (os comentários muito curtos, como "Great stay", não têm informação temática). 30 000 documentos chegam para tópicos estáveis num vocabulário deste tamanho e permitem testar vários valores de *k* em tempo útil. Os modelos ajustados são depois aplicados (`transform`) a todo o Nível 2.

O conjunto de **validação manual** do sentimento é retirado do Nível 2 e definido na secção 3.

In [ ]:
N_PER_LISTING = 20
N_MODEL_SAMPLE = 30_000
MIN_TOPIC_TOKENS = 5

# Nível 2: 20 reviews inglesas mais recentes por anúncio
feat = (en.sort_values(["listing_id", "date"], ascending=[True, False])
          .groupby("listing_id").head(N_PER_LISTING).copy())
feat["text_topic"] = [normalize_for_topics(t, l) for t, l in zip(feat["text_clean"], feat["listing_id"])]
feat["n_topic_tokens"] = feat["text_topic"].str.split().str.len()

# Nível 3: amostra aleatória para ajustar os modelos de tópicos
eligible = feat[feat["n_topic_tokens"] >= MIN_TOPIC_TOKENS]
model_ids = eligible.sample(N_MODEL_SAMPLE, random_state=RANDOM_STATE)["id"]

en["in_feature_corpus"] = en["id"].isin(feat["id"])
en["in_model_sample"] = en["id"].isin(model_ids)

per_listing_en = en.groupby("listing_id").size()
levels = pd.DataFrame({
    "n_reviews": [len(en), len(feat), len(model_ids)],
    "n_anúncios": [en["listing_id"].nunique(), feat["listing_id"].nunique(),
                   en.loc[en["in_model_sample"], "listing_id"].nunique()],
    "uso": ["VADER", "RoBERTa + atribuição de tópicos + variáveis (secção 6)", "Ajuste LDA/BERTopic, escolha de k, coerência"],
}, index=["Nível 1 — corpus inglês", "Nível 2 — corpus de variáveis", "Nível 3 — amostra de modelação"])
display(levels)
print(f"Anúncios com todas as reviews inglesas incluídas no Nível 2: {(per_listing_en <= N_PER_LISTING).mean():.1%}")
print(f"Elegíveis para o Nível 3 (>= {MIN_TOPIC_TOKENS} tokens): {len(eligible):,} ({len(eligible) / len(feat):.1%} do Nível 2)")

In [ ]:
# Representatividade: os níveis amostrados mantêm o perfil do corpus completo?
def profile(d):
    return pd.Series({
        "palavras (média)": d["n_words"].mean(),
        "palavras (mediana)": d["n_words"].median(),
        "% reviews desde 2023": (d["date"].dt.year >= 2023).mean() * 100,
        "% reviews até 2019": (d["date"].dt.year <= 2019).mean() * 100,
    })

pd.DataFrame({
    "Nível 1": profile(en),
    "Nível 2": profile(feat),
    "Nível 3": profile(en[en["in_model_sample"]]),
}).round(1)

Diferenças esperadas e assumidas:
- O **Nível 2** é mais recente que o corpus completo, por construção (reviews mais recentes por anúncio).
- O **Nível 3** tem comentários mais longos, porque exclui os que têm menos de 5 tokens úteis. É o desejado para a modelação de tópicos: os comentários curtos não têm conteúdo temático.

In [ ]:
# Exportar ficheiros intermédios (entregáveis da Parte I)
cols = ["id", "listing_id", "date", "reviewer_id", "text_clean", "n_words", "language", "language_raw",
        "in_feature_corpus", "in_model_sample"]
out = en[cols].merge(feat[["id", "text_topic", "n_topic_tokens"]], on="id", how="left")
out.to_parquet(INT_DIR / "reviews_en_clean.parquet", index=False)
print(f"Guardado: {INT_DIR / 'reviews_en_clean.parquet'}  ({len(out):,} linhas)")
show_funnel(funnel)

### 2.6 Síntese da secção 2
**Funil de dados:** 760 023 reviews originais → **680 216 reviews em inglês** (89,5%), cobrindo **22 108 anúncios**.

| Passo | Removidas | Motivo |
|---|---|---|
| Comentários vazios | 158 | Sem texto |
| Sem letras | 2 068 | ".", "-", emojis: sem conteúdo para PLN |
| Duplicados reais | 227 | Mesmo hóspede, anúncio e texto |
| Não ingleses | 77 354 | Ferramentas de PLN otimizadas para inglês; evitar tópicos por idioma |

**Idiomas.** O corpus é dominado pelo **inglês (89,8%)**, seguido de **alemão (5,5%)**, **francês (1,5%)** e **neerlandês (1,0%)**, o que reflete os principais mercados emissores de turistas para Cape Town. Os idiomas locais (afrikaans, zulu, xhosa) quase não aparecem. A deteção (lingua) é fiável em textos com conteúdo, mas falha em comentários de uma ou duas palavras. Corrigimo-la com uma regra baseada no vocabulário inglês, que **reclassificou 4 834 comentários** ("Good" → somali, "Nice" → checo, "Perfect" → latim). Com o filtro de idioma só **214 anúncios** perdem todas as reviews.

**Normalização em dois níveis.** O `text_clean` (limpeza mínima, que mantém pontuação, maiúsculas, emojis e negações) alimenta o sentimento e os embeddings. O `text_topic` (minúsculas, sem *stopwords* gerais e de domínio, sem nomes de anfitriões, lematizado) alimenta o LDA.

**Amostragem em três níveis:**
- **Nível 1:** 680 216 reviews → VADER;
- **Nível 2:** 258 094 reviews (as 20 mais recentes por anúncio; 100% dos anúncios; 63% dos anúncios ficam com todas as reviews) → RoBERTa, atribuição de tópicos e variáveis por anúncio;
- **Nível 3:** 30 000 reviews com pelo menos 5 tokens úteis → ajuste do LDA e do BERTopic.

O Nível 2 é mais recente que o corpus completo (89% das reviews desde 2023, contra 64%). Isto é **intencional**: as variáveis por anúncio devem refletir o estado atual do anúncio.

**Ficheiros intermédios:** `outputs/intermediate/reviews_language.parquet` (idioma por review) e `outputs/intermediate/reviews_en_clean.parquet` (corpus inglês limpo, com indicadores de nível).

## 3. Análise de Sentimentos

Comparamos duas famílias de ferramentas:

| | **VADER** | **RoBERTa** (`cardiffnlp/twitter-roberta-base-sentiment-latest`) |
|---|---|---|
| Tipo | Léxico + regras (Hutto & Gilbert, 2014) | Transformer (RoBERTa-base) treinado em ~124M tweets e afinado para sentimento (Loureiro et al., 2022) |
| Saída | `compound` contínuo em [-1, 1] + proporções pos/neu/neg | Probabilidades das classes negativo/neutro/positivo |
| Contexto | Regras para negação, intensificadores, maiúsculas, "but" | Representação contextual da frase completa |
| Custo | ~10 000 comentários/s (CPU) | ~100 comentários/s (GPU MPS) |
| Aplicado a | Nível 1 (680k reviews) | Nível 2 (258k reviews) |

Para comparar as ferramentas na mesma escala definimos:
- **Rótulo** (positivo/neutro/negativo): VADER com os limiares recomendados pelos autores (`compound ≥ 0,05` → positivo, `≤ -0,05` → negativo); RoBERTa pela classe mais provável;
- **Score contínuo** em [-1, 1]: o `compound` no VADER e `P(positivo) − P(negativo)` no RoBERTa.

Ambas as ferramentas usam o `text_clean`, que mantém negações, pontuação e emojis.

### 3.1 VADER

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

VADER_CACHE = INT_DIR / "sentiment_vader.parquet"

def to_label(score, thr=0.05):
    return np.select([score >= thr, score <= -thr], ["positivo", "negativo"], "neutro")

if VADER_CACHE.exists():
    vader = pd.read_parquet(VADER_CACHE)
else:
    sia = SentimentIntensityAnalyzer()
    vs = pd.DataFrame([sia.polarity_scores(t) for t in en["text_clean"]])
    vader = pd.DataFrame({"id": en["id"].values, "vader_compound": vs["compound"].values,
                          "vader_pos": vs["pos"].values, "vader_neg": vs["neg"].values})
    vader.to_parquet(VADER_CACHE, index=False)

en = en.drop(columns=[c for c in en.columns if c.startswith("vader_")]).merge(vader, on="id", how="left")
en["vader_label"] = to_label(en["vader_compound"])

LABELS = ["negativo", "neutro", "positivo"]
display(en["vader_label"].value_counts(normalize=True).reindex(LABELS).mul(100).round(2).to_frame("% reviews (Nível 1)"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(en["vader_compound"], bins=80, ax=ax)
ax.axvline(0.05, color="black", ls=":"); ax.axvline(-0.05, color="black", ls=":")
ax.set(title="Distribuição do VADER compound (Nível 1)", xlabel="compound", ylabel="Nº de reviews")
save_fig("p1_vader_distribution")
plt.show()

### 3.2 RoBERTa pré-treinado
Escolhemos o modelo da Cardiff NLP porque:
- tem **três classes** (incluindo neutro), o que permite comparar diretamente com o VADER;
- foi treinado em **texto curto e informal de redes sociais**, um registo próximo das reviews Airbnb;
- é um dos modelos de sentimento em inglês mais usados no Hugging Face (*benchmark* TweetEval).

Detalhes de implementação:
- **truncagem a 256 tokens**, que cobre mais de 99% dos comentários sem cortes (o percentil 99 é ~174 palavras);
- os comentários são **ordenados por comprimento** dentro de cada lote, o que reduz o *padding* e acelera a inferência ~2,5×;
- o processamento é feito em **blocos de 25 000** com *checkpoint* em disco, para que uma interrupção não obrigue a recomeçar do zero.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ROBERTA = "cardiffnlp/twitter-roberta-base-sentiment-latest"
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
ROB_CACHE = INT_DIR / "sentiment_roberta.parquet"
CHUNK_DIR = INT_DIR / "roberta_chunks"

def roberta_probs(texts, tok, model, batch_size=64, max_length=256):
    """Probabilidades [neg, neu, pos] por texto. Ordena por comprimento para minimizar padding."""
    order = np.argsort([len(t) for t in texts])
    probs = np.zeros((len(texts), 3), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            idx = order[start:start + batch_size]
            enc = tok([texts[i] for i in idx], padding=True, truncation=True,
                      max_length=max_length, return_tensors="pt").to(DEVICE)
            probs[idx] = torch.softmax(model(**enc).logits, dim=-1).float().cpu().numpy()
    return probs

if ROB_CACHE.exists():
    rob = pd.read_parquet(ROB_CACHE)
else:
    tok = AutoTokenizer.from_pretrained(ROBERTA)
    model = AutoModelForSequenceClassification.from_pretrained(ROBERTA).to(DEVICE).eval()
    assert [model.config.id2label[i].lower() for i in range(3)] == ["negative", "neutral", "positive"]
    CHUNK_DIR.mkdir(exist_ok=True)
    ids, texts = feat["id"].to_numpy(), feat["text_clean"].tolist()
    CHUNK = 25_000
    for k, start in enumerate(range(0, len(texts), CHUNK)):
        path = CHUNK_DIR / f"chunk_{k:03d}.parquet"
        if path.exists():
            continue
        p = roberta_probs(texts[start:start + CHUNK], tok, model)
        pd.DataFrame({"id": ids[start:start + CHUNK], "rob_neg": p[:, 0], "rob_neu": p[:, 1], "rob_pos": p[:, 2]}).to_parquet(path, index=False)
        print(f"bloco {k}: {min(start + CHUNK, len(texts)):,}/{len(texts):,}", flush=True)
    rob = pd.concat([pd.read_parquet(f) for f in sorted(CHUNK_DIR.glob("chunk_*.parquet"))], ignore_index=True)
    rob.to_parquet(ROB_CACHE, index=False)

feat = feat.drop(columns=[c for c in feat.columns if c.startswith(("rob_", "vader_"))])
feat = feat.merge(rob, on="id", how="left").merge(en[["id", "vader_compound", "vader_label"]], on="id", how="left")
feat["rob_score"] = feat["rob_pos"] - feat["rob_neg"]
feat["rob_label"] = np.array(LABELS)[feat[["rob_neg", "rob_neu", "rob_pos"]].to_numpy().argmax(axis=1)]
print(f"Reviews com score RoBERTa: {feat['rob_score'].notna().sum():,}")

pd.DataFrame({
    "VADER (Nível 2)": feat["vader_label"].value_counts(normalize=True),
    "RoBERTa (Nível 2)": feat["rob_label"].value_counts(normalize=True),
}).reindex(LABELS).mul(100).round(2)

### 3.3 Comparação entre ferramentas

In [ ]:
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr, pearsonr

agree = (feat["vader_label"] == feat["rob_label"]).mean()
kappa = cohen_kappa_score(feat["vader_label"], feat["rob_label"])
rho = spearmanr(feat["vader_compound"], feat["rob_score"]).statistic
r = pearsonr(feat["vader_compound"], feat["rob_score"]).statistic
print(f"Concordância de rótulos: {agree:.1%} | Cohen's kappa: {kappa:.3f}")
print(f"Correlação entre scores: Spearman = {rho:.3f} | Pearson = {r:.3f}")

ct = pd.crosstab(feat["vader_label"], feat["rob_label"], normalize="all").reindex(index=LABELS, columns=LABELS) * 100
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.heatmap(ct, annot=True, fmt=".2f", cmap="Blues", ax=axes[0], cbar=False)
axes[0].set(title="Matriz de concordância (% do Nível 2)", xlabel="RoBERTa", ylabel="VADER")
axes[1].hexbin(feat["vader_compound"], feat["rob_score"], gridsize=50, bins="log", cmap="viridis")
axes[1].set(title="VADER compound vs RoBERTa score (escala log)", xlabel="VADER compound", ylabel="RoBERTa P(pos) − P(neg)")
save_fig("p1_sentiment_agreement")
plt.show()

In [ ]:
# A concordância depende do comprimento do comentário?
len_bins = pd.cut(feat["n_words"], [0, 5, 15, 30, 60, 120, np.inf], labels=["1-5", "6-15", "16-30", "31-60", "61-120", "120+"])
by_len = feat.groupby(len_bins, observed=True).apply(lambda d: pd.Series({
    "n": len(d),
    "concordância %": (d["vader_label"] == d["rob_label"]).mean() * 100,
    "% neg VADER": (d["vader_label"] == "negativo").mean() * 100,
    "% neg RoBERTa": (d["rob_label"] == "negativo").mean() * 100,
}))
by_len.round(2)

### 3.4 Exemplos representativos e casos de desacordo
Sem rótulos de referência, uma forma de avaliar as ferramentas é ler os casos em que discordam. Mostramos os desacordos **mais fortes** (maior diferença entre os dois scores) em cada direção.

In [ ]:
def show_examples(d, n=6, sort_col=None, ascending=False):
    d = d.sort_values(sort_col, ascending=ascending) if sort_col else d.sample(min(n, len(d)), random_state=RANDOM_STATE)
    out = d.head(n)[["text_clean", "vader_compound", "rob_score", "vader_label", "rob_label"]].copy()
    out["text_clean"] = out["text_clean"].str[:350]
    return out.round(3)

feat["score_gap"] = feat["vader_compound"] - feat["rob_score"]
print("VADER positivo / RoBERTa negativo:", ((feat["vader_label"] == "positivo") & (feat["rob_label"] == "negativo")).sum())
show_examples(feat[(feat["vader_label"] == "positivo") & (feat["rob_label"] == "negativo")], sort_col="score_gap")

In [ ]:
print("VADER negativo / RoBERTa positivo:", ((feat["vader_label"] == "negativo") & (feat["rob_label"] == "positivo")).sum())
show_examples(feat[(feat["vader_label"] == "negativo") & (feat["rob_label"] == "positivo")], sort_col="score_gap", ascending=True)

In [ ]:
print("VADER neutro / RoBERTa positivo (amostra aleatória):")
show_examples(feat[(feat["vader_label"] == "neutro") & (feat["rob_label"] == "positivo")])

In [ ]:
print("Ambos negativos (amostra aleatória): exemplos de comentários claramente negativos")
show_examples(feat[(feat["vader_label"] == "negativo") & (feat["rob_label"] == "negativo")])

### 3.5 Validação indireta: sentimento vs pontuação do anúncio
Se as ferramentas medirem bem a satisfação, o **sentimento médio de um anúncio** deve estar correlacionado com o `review_scores_rating` atribuído pelos hóspedes. Usamos os anúncios com pelo menos 5 reviews no Nível 2.

*Nota:* aqui o `review_scores_rating` serve só como **validação externa** das ferramentas. A relação com a variável alvo, e o risco de *data leakage*, são discutidos na Parte II.

In [ ]:
agg = (feat.groupby("listing_id")
           .agg(n=("id", "size"), vader_mean=("vader_compound", "mean"), rob_mean=("rob_score", "mean"),
                vader_neg=("vader_label", lambda s: (s == "negativo").mean()),
                rob_neg=("rob_label", lambda s: (s == "negativo").mean()))
           .join(listings.set_index("id")["review_scores_rating"]))
agg = agg[(agg["n"] >= 5) & agg["review_scores_rating"].notna()]

val = pd.DataFrame({
    "Spearman com rating": [spearmanr(agg[c], agg["review_scores_rating"]).statistic
                            for c in ["vader_mean", "rob_mean", "vader_neg", "rob_neg"]],
}, index=["VADER — score médio", "RoBERTa — score médio", "VADER — % negativos", "RoBERTa — % negativos"])
print(f"Anúncios usados: {len(agg):,}")
display(val.round(3))

rating_bins = pd.cut(agg["review_scores_rating"], [0, 4.0, 4.5, 4.7, 4.8, 4.9, 5.0])
fig, ax = plt.subplots(figsize=(10, 4))
agg.groupby(rating_bins, observed=True)[["vader_mean", "rob_mean"]].mean().plot(kind="bar", ax=ax)
ax.set(title="Sentimento médio do anúncio por faixa de rating", xlabel="review_scores_rating", ylabel="Score médio")
ax.legend(["VADER", "RoBERTa"]); ax.tick_params(axis="x", rotation=0)
save_fig("p1_sentiment_vs_rating")
plt.show()

### 3.6 Conjunto de validação manual
Para medir o desempenho de cada ferramenta com rótulos de referência, o grupo anota manualmente um conjunto de reviews.

**Dimensão: 300 reviews**, em dois estratos:
| Estrato | n | Como é selecionado | Para quê |
|---|---|---|---|
| A — aleatório | 150 | Amostra aleatória simples do Nível 2 | Estimativa **não enviesada** da accuracy no corpus real. Com n = 150 e accuracy esperada de ~85%, o intervalo de confiança a 95% tem ±5,7 pp: precisão suficiente para distinguir as ferramentas se diferirem 10 pp ou mais |
| B — direcionado | 150 | 75 casos de desacordo VADER/RoBERTa + 75 casos que pelo menos uma ferramenta classifica como negativo | Com ~3% de negativos, o estrato A teria só ~5 negativos. Este estrato garante exemplos suficientes para avaliar a **classe negativa**, a mais importante para as variáveis da secção 6 |

Porquê 300 e não mais? Cada anotador precisa de ~1 a 1,5 horas para anotar 300 reviews curtas (mediana de 28 palavras). **Todos os membros anotam as mesmas 300**, o que permite medir a **concordância entre anotadores** (kappa de Cohen). O rótulo de referência é o **voto maioritário**.

**Protocolo (anotação cega):** o ficheiro `annotation/validacao_sentimento.xlsx` tem só o texto, por ordem aleatória, sem as previsões das ferramentas nem o estrato. A ligação ao estrato fica em `outputs/intermediate/annotation_key.parquet`. As regras de anotação estão na folha "Instruções" do ficheiro.

A célula seguinte **só cria o ficheiro se ainda não existir**, para não apagar anotações já feitas.

In [ ]:
ANN_DIR = Path("annotation")
ANN_FILE = ANN_DIR / "validacao_sentimento.xlsx"
ANN_KEY = INT_DIR / "annotation_key.parquet"
ANNOTATORS = ["anotador_1", "anotador_2", "anotador_3"]

if not ANN_FILE.exists():
    from openpyxl import load_workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation

    rng = np.random.default_rng(RANDOM_STATE)
    pool = feat[feat["n_words"] >= 3]
    a = pool.sample(150, random_state=RANDOM_STATE)
    rest = pool.drop(a.index)
    b1 = rest[rest["vader_label"] != rest["rob_label"]].sample(75, random_state=RANDOM_STATE)
    rest = rest.drop(b1.index)
    b2 = rest[(rest["vader_label"] == "negativo") | (rest["rob_label"] == "negativo")].sample(75, random_state=RANDOM_STATE)
    sel = pd.concat([a.assign(estrato="A_aleatorio"), b1.assign(estrato="B_desacordo"), b2.assign(estrato="B_negativo")])
    sel = sel.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    sel["ann_id"] = np.arange(1, len(sel) + 1)

    sel[["ann_id", "id", "listing_id", "estrato"]].to_parquet(ANN_KEY, index=False)

    sheet = sel[["ann_id", "text_clean"]].rename(columns={"text_clean": "comentario"})
    for c in ANNOTATORS:
        sheet[c] = ""
    sheet["notas"] = ""

    instructions = pd.DataFrame({"Instruções de anotação": [
        "Objetivo: classificar o SENTIMENTO GLOBAL do hóspede em relação à estadia.",
        "Cada membro do grupo preenche apenas a sua coluna (anotador_1, anotador_2, anotador_3), SEM ver as colunas dos colegas.",
        "Valores possíveis (lista pendente): positivo / neutro / negativo.",
        "",
        "positivo — o hóspede está globalmente satisfeito e recomendaria (mesmo com pequenas críticas: 'Great place, wifi a bit slow').",
        "negativo — o hóspede está globalmente insatisfeito ou relata problemas relevantes (limpeza, segurança, anfitrião, não corresponde ao anúncio), mesmo com tom educado.",
        "neutro — comentário factual sem opinião clara, ou elogios e críticas equilibrados sem sentimento predominante.",
        "",
        "Regras: 1) Avaliar o comentário inteiro, não frases isoladas. 2) Em comentários mistos, escolher o sentimento predominante. "
        "3) Ironia/sarcasmo: anotar o sentido real. 4) Não usar as ferramentas automáticas nem consultar colegas durante a anotação.",
        "Use a coluna 'notas' para casos duvidosos.",
        "Se o grupo tiver só 2 membros, deixar anotador_3 vazio.",
    ]})

    ANN_DIR.mkdir(exist_ok=True)
    with pd.ExcelWriter(ANN_FILE, engine="openpyxl") as xw:
        instructions.to_excel(xw, sheet_name="Instruções", index=False)
        sheet.to_excel(xw, sheet_name="Anotação", index=False)

    # Formatação: largura das colunas, quebra de linha e lista pendente para os rótulos
    wb = load_workbook(ANN_FILE)
    ws_i, ws = wb["Instruções"], wb["Anotação"]
    ws_i.column_dimensions["A"].width = 140
    for row in ws_i.iter_rows(min_row=2):
        row[0].alignment = Alignment(wrap_text=True, vertical="top")
    widths = {"A": 8, "B": 100, "C": 14, "D": 14, "E": 14, "F": 30}
    for col, w in widths.items():
        ws.column_dimensions[col].width = w
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="2F5597")
    for row in ws.iter_rows(min_row=2):
        row[1].alignment = Alignment(wrap_text=True, vertical="top")
    dv = DataValidation(type="list", formula1='"positivo,neutro,negativo"', allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f"C2:E{ws.max_row}")
    ws.freeze_panes = "C2"
    wb.save(ANN_FILE)
    print(f"Ficheiro de anotação criado: {ANN_FILE} ({len(sel)} reviews)")
else:
    print(f"Ficheiro de anotação já existe: {ANN_FILE} (não foi alterado)")

pd.read_parquet(ANN_KEY)["estrato"].value_counts()

In [ ]:
# Avaliação face às anotações manuais (corre quando o ficheiro estiver preenchido)
from sklearn.metrics import accuracy_score, f1_score, classification_report

ann = pd.read_excel(ANN_FILE, sheet_name="Anotação").merge(pd.read_parquet(ANN_KEY), on="ann_id")
for c in ANNOTATORS:
    ann[c] = ann[c].astype("string").str.strip().str.lower().replace("", pd.NA)
used = [c for c in ANNOTATORS if ann[c].notna().sum() > 0]

if len(used) < 2 or ann[used].isna().any().any():
    print(f"Anotação incompleta: {ann[used].notna().all(axis=1).sum() if used else 0}/{len(ann)} reviews "
          f"anotadas por todos os anotadores ativos ({used or 'nenhum'}). São precisos pelo menos 2 anotadores.")
else:
    # Concordância entre anotadores
    from itertools import combinations
    for x, y in combinations(used, 2):
        print(f"Kappa {x} vs {y}: {cohen_kappa_score(ann[x], ann[y]):.3f}")

    # Rótulo de referência = voto maioritário (empates com 2 anotadores são excluídos)
    def majority(row):
        vc = row.value_counts()
        return vc.index[0] if vc.iloc[0] > len(row) / 2 else pd.NA
    ann["gold"] = ann[used].apply(majority, axis=1)
    print(f"Reviews com rótulo de referência (maioria): {ann['gold'].notna().sum()}/{len(ann)}")

    ann = ann.merge(feat[["id", "vader_label", "rob_label"]], on="id")
    g = ann.dropna(subset=["gold"])
    res = []
    for name, d in [("Estrato A (aleatório)", g[g["estrato"] == "A_aleatorio"]), ("Total (A + B)", g)]:
        for tool, col in [("VADER", "vader_label"), ("RoBERTa", "rob_label")]:
            res.append({"conjunto": name, "ferramenta": tool, "n": len(d),
                        "accuracy": accuracy_score(d["gold"], d[col]),
                        "F1 macro": f1_score(d["gold"], d[col], average="macro"),
                        "F1 negativo": f1_score(d["gold"], d[col], labels=["negativo"], average="macro")})
    display(pd.DataFrame(res).round(3))
    for tool, col in [("VADER", "vader_label"), ("RoBERTa", "rob_label")]:
        print(f"\n{tool} — total:\n", classification_report(g["gold"], g[col], labels=LABELS, zero_division=0))

In [ ]:
# Guardar o sentimento por review (Nível 2) para as secções 4 e 6
feat[["id", "listing_id", "date", "vader_compound", "vader_label", "rob_neg", "rob_neu", "rob_pos",
      "rob_score", "rob_label"]].to_parquet(INT_DIR / "sentiment_level2.parquet", index=False)
print("Guardado:", INT_DIR / "sentiment_level2.parquet")

### 3.7 Diferenças e limitações das ferramentas
__DISCUSSION_3__

## 4. Modelação de Tópicos

- [ ] LDA
- [ ] BERTopic
- [ ] Escolha do número de tópicos
- [ ] Coerência + avaliação qualitativa
- [ ] Tópicos × sentimento: temas com mais comentários negativos

_Por fazer._

## 5. Comparação com a Literatura

- [ ] Literatura técnica / metodológica
- [ ] Literatura temática (fatores de satisfação)
- [ ] Convergências e divergências com os resultados

_Por fazer._

## 6. Construção das Variáveis de Texto

- [ ] Agregação por listing (sentimento médio/dispersão, % negativos, nº/comprimento reviews, tópicos, tendência)
- [ ] Justificação das variáveis e do método de agregação
- [ ] Exportar tabela para outputs/intermediate/

_Por fazer._

---
# PARTE II — Modelos supervisionados de classificação de anúncios

## 1. Integração e Preparação dos Dados

- [ ] Integrar listings + variáveis de texto
- [ ] Tratamento de listings sem reviews/comentários
- [ ] Definir target high_quality (limiares justificados)
- [ ] Nº de anúncios alta vs baixa qualidade; balanceamento
- [ ] Preço médio por classe
- [ ] Seleção de colunas
- [ ] Discussão de data leakage

_Por fazer._

## 2. Modelo Naive Bayes

- [ ] Treino
- [ ] Accuracy, precisão, recall, F1
- [ ] AUC-ROC
- [ ] Alta qualidade corretamente identificados
- [ ] Variável mais influente
- [ ] Efeito do preço
- [ ] Efeito do nº de reviews (quantificado)

_Por fazer._

## 3. Modelo Random Forest

- [ ] Treino
- [ ] Métricas
- [ ] Feature importance; top 5 (alguma da Parte I?)
- [ ] AUC-ROC vs Naive Bayes
- [ ] Robustez a overfitting

_Por fazer._

## 4. Modelo SVM

- [ ] Kernels linear, RBF, polinomial
- [ ] Métricas
- [ ] Melhor kernel; AUC-ROC vs NB e RF

_Por fazer._

## 5. Contributo das Variáveis de Texto

- [ ] Retreinar os 3 modelos sem variáveis da Parte I
- [ ] Comparação com/sem texto
- [ ] Ganho e interpretação

_Por fazer._

## 6. Conclusão e Comparação de Modelos

- [ ] Tabela-resumo (com/sem texto)
- [ ] Modelo recomendado (recall classe positiva, FN vs FP)
- [ ] Parágrafo de conclusão (máx. 500 palavras)

_Por fazer._

---
# Conclusões Gerais

- [ ] Variáveis de texto vs variáveis mais relevantes na Parte II; confronto com a literatura
- [ ] Limitações e potenciais melhorias

_Por fazer._